# 📝 EXAMEN · Estructuras de control
## RA5 · Excepciones y código robusto

**Alumno/a:** Darío Vega Ruiz  **Grupo:** Python  **Fecha:** 25/11/2026

**Resultado de aprendizaje:** *Crea arquitectura de código de forma eficiente y escribe código robusto.*

### Instrucciones
- Completa las celdas marcadas con `# TU CÓDIGO AQUÍ` o `# TU RESPUESTA AQUÍ`. No modifiques las celdas de **comprobación**.
- Las celdas de comprobación son una ayuda automática; la calificación final la decide el profesorado según los criterios indicados en cada ejercicio.
- Ejecuta el cuaderno de arriba abajo (*Kernel → Restart & Run All*) antes de entregarlo.
- Duración orientativa: 60 minutos. Puntuación total: **10 puntos**.

### Cobertura de los criterios de evaluación
Cada criterio se evalúa **una única vez**.

| Criterio | Ejercicio | Evidencia que se evalúa |
|---|---|---|
| a) Diferenciar excepción y error de sintaxis | **1** | Clasifica correctamente fragmentos de código |
| b) Escribir instrucciones de captura de excepciones | **2** | Usa `try/except/else/finally` con sintaxis correcta |
| c) Capturar y tratar excepciones | **2** | Captura excepciones concretas y continúa con una estrategia clara |
| d) Tratar excepciones | **2** | Lanza (`raise`) una excepción propia y la gestiona en el llamador |
| e) Depurar excepciones | **3** | Identifica tipo, línea y causa de excepciones a partir del *traceback* |
| f) Escribir código robusto con las sentencias adecuadas | **3** | Reescribe la función con `with`, validación y manejo de errores |

| Ejercicio | Criterios | Puntos |
|---|---|---|
| 1 | a | 2 |
| 2 | b, c, d | 4 |
| 3 | e, f | 4 |

In [61]:
# Celda de utilidades (NO modificar)
import hashlib
def _h(x):
    return hashlib.sha256(repr(x).encode()).hexdigest()[:8]
print("Listo")

Listo


---
## Ejercicio 1 · Criterio evaluado: **a)** · 2 puntos

Clasifica cada fragmento de código. Escribe `"sintaxis"` si Python lo rechaza **antes de ejecutarlo** (error de sintaxis), o el **nombre de la excepción** (`"ValueError"`, `"ZeroDivisionError"`, ...) si es válido pero falla **durante la ejecución**.

| Nº | Fragmento |
|---|---|
| 1 | `print("Hola"` |
| 2 | `int("abc")` |
| 3 | `10 / 0` |
| 4 | `[1, 2, 3][3]` |
| 5 | `for i in range(3)` ⏎ `    print(i)` *(falta algo en la primera línea)* |
| 6 | `"5" + 5` |

In [62]:
clasificacion = {
   1: "sintaxis",
    2: "ValueError",
    3: "ZeroDivisionError",
    4: "IndexError",
    5: "sintaxis",
    6: "TypeError",
}

In [63]:
# Comprobación (NO modificar)
esperado = {1: "bb691598", 2: "3200aae0", 3: "90e4dc48",
            4: "ac493dfe", 5: "bb691598", 6: "64efeab0"}
for k, h in esperado.items():
    assert _h(clasificacion[k].strip()) == h, f"Revisa la respuesta {k}"
print("✔ Ejercicio 1 superado")

✔ Ejercicio 1 superado


---
## Ejercicio 2 · Criterios evaluados: **b)**, **c)** y **d)** · 4 puntos

Se reciben edades como texto (`"25"`, `"x"`, `None`, `"130"`...). Implementa:

**2.1 (criterio d)** La excepción propia `EdadInvalidaError` (hereda de `Exception`) y la función `leer_edad(texto)`, que:
- convierte `texto` a entero;
- si no se puede convertir (`ValueError` o `TypeError`) **lanza** `EdadInvalidaError` con un mensaje claro;
- si el número no está entre 0 y 120 **lanza** `EdadInvalidaError`;
- en otro caso devuelve la edad.

**2.2 (criterios b y c)** La función `procesar_edades(lista)`, que recorre la lista, **captura** `EdadInvalidaError` y devuelve un diccionario:

```python
{"validas": [...edades correctas...], "errores": <nº de edades inválidas>, "procesados": <nº total de elementos intentados>}
```

Usa la estructura **`try / except / else / finally`**: añade la edad en el `else`, cuenta el error en el `except` y cuenta los procesados en el `finally`.

In [64]:
class EdadInvalidaError(Exception):
    pass

def leer_edad(texto):
    try:
        edad = int(texto)
    except (ValueError, TypeError):
        raise EdadInvalidaError("La edad debe ser un número entero")

    if edad < 0 or edad > 120:
        raise EdadInvalidaError("La edad debe estar entre 0 y 120")

    return edad

def procesar_edades(lista):
    resultado = {"validas": [], "errores": 0, "procesados": 0}

    for texto in lista:
        try:
            edad = leer_edad(texto)
        except EdadInvalidaError:
            resultado["errores"] += 1
        else:
            resultado["validas"].append(edad)
        finally:
            resultado["procesados"] += 1

    return resultado

In [65]:
# Comprobación (NO modificar)
assert issubclass(EdadInvalidaError, Exception)
assert leer_edad("25") == 25 and leer_edad(0) == 0 and leer_edad("120") == 120
for malo in ["x", None, "130", "-1", "3.5"]:
    try:
        leer_edad(malo)
        raise AssertionError(f"leer_edad({malo!r}) debería lanzar EdadInvalidaError")
    except EdadInvalidaError as e:
        assert str(e).strip(), "El mensaje de error no puede estar vacío"

assert procesar_edades(["25", "x", None, "130", "40"]) == {"validas": [25, 40], "errores": 3, "procesados": 5}
assert procesar_edades([]) == {"validas": [], "errores": 0, "procesados": 0}
print("✔ Ejercicio 2 superado (el profesorado comprueba el uso de try/except/else/finally)")

✔ Ejercicio 2 superado (el profesorado comprueba el uso de try/except/else/finally)


---
## Ejercicio 3 · Criterios evaluados: **e)** y **f)** · 4 puntos

Esta función calcula la **media de los números de un fichero** (uno por línea), pero falla con algunos ficheros. Primero se crean tres ficheros de prueba:

In [66]:
# Preparación (NO modificar)
with open("datos_ok.txt", "w") as f:
    f.write("4\n6\n8\n")
with open("datos_mal.txt", "w") as f:
    f.write("4\nx\n8\n\n")
with open("vacio.txt", "w") as f:
    f.write("")

def media_fichero(ruta):
    f = open(ruta)
    total = 0
    n = 0
    for linea in f:
        total += float(linea)
        n += 1
    f.close()
    return total / n

print(media_fichero("datos_ok.txt"))

6.0


**3.1 (criterio e, depuración)** Ejecuta la celda siguiente (lanza la función con cada fichero y muestra el *traceback* de cada fallo) y úsala para completar el diagnóstico.

In [67]:
import traceback
for ruta in ["datos_mal.txt", "no_existe.txt", "vacio.txt"]:
    print("=" * 50, ruta)
    try:
        media_fichero(ruta)
    except Exception:
        traceback.print_exc()

================================================== datos_mal.txt
================================================== no_existe.txt
================================================== vacio.txt


Traceback (most recent call last):
  File "C:\Users\dvegrui2207\AppData\Local\Temp\ipykernel_2112\2412219902.py", line 5, in <module>
    media_fichero(ruta)
  File "C:\Users\dvegrui2207\AppData\Local\Temp\ipykernel_2112\3511825170.py", line 14, in media_fichero
    total += float(linea)
             ^^^^^^^^^^^^
ValueError: could not convert string to float: 'x\n'
Traceback (most recent call last):
  File "C:\Users\dvegrui2207\AppData\Local\Temp\ipykernel_2112\2412219902.py", line 5, in <module>
    media_fichero(ruta)
  File "C:\Users\dvegrui2207\AppData\Local\Temp\ipykernel_2112\3511825170.py", line 10, in media_fichero
    f = open(ruta)
        ^^^^^^^^^^
  File "\\?\C:\Users\dvegrui2207\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\IPython\core\interactiveshell.py", line 350, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'no_existe.txt'
Traceback (mos

In [68]:
diagnostico = {
    "datos_mal.txt": "ValueError",     # nombre de la excepción que se produce
    "no_existe.txt": "FileNotFoundError",
    "vacio.txt": "ZeroDivisionError",
    "linea_que_falla_con_datos_mal": "total += float(linea)",   # copia la línea de código donde se produce el error con datos_mal.txt
}

**3.2 (criterio f, código robusto)** Escribe `media_fichero_robusta(ruta)` con estas reglas:

- Abre el fichero con **`with`** (se cierra siempre, aunque haya errores).
- **Ignora** las líneas vacías o que no sean números (sin detener el programa).
- Si el fichero **no existe**, devuelve `None`.
- Si no hay **ningún** número válido, devuelve `None` (no debe lanzar `ZeroDivisionError`).
- En otro caso devuelve la media.

In [69]:
def media_fichero_robusta(ruta):
    total = 0
    numeros = 0

    try:
        with open(ruta, "r") as fichero:
            for linea in fichero:
                try:
                    numero = float(linea)
                    total += numero
                    numeros += 1
                except ValueError:
                    pass
    except FileNotFoundError:
        return None

    if numeros == 0:
        return None

    return total / numeros

In [70]:
# Comprobación (NO modificar)
for ruta, h in {"datos_mal.txt": "3200aae0", "no_existe.txt": "d1dc0e4a", "vacio.txt": "90e4dc48"}.items():
    assert _h(diagnostico[ruta].strip()) == h, f"3.1: revisa la excepción de {ruta}"
assert "float" in diagnostico["linea_que_falla_con_datos_mal"], "3.1: revisa la línea que falla"

assert media_fichero_robusta("datos_ok.txt") == 6.0
assert media_fichero_robusta("datos_mal.txt") == 6.0, "Debe ignorar 'x' y las líneas vacías"
assert media_fichero_robusta("vacio.txt") is None
assert media_fichero_robusta("no_existe.txt") is None
print("✔ Ejercicio 3 superado (el profesorado comprueba el uso de with y la gestión de errores)")

✔ Ejercicio 3 superado (el profesorado comprueba el uso de with y la gestión de errores)


---
## Hoja de corrección (profesorado)

| Ejercicio | Criterios | Máx. | Obtenido | Observaciones |
|---|---|---|---|---|
| 1 | a | 2 | | |
| 2 | b, c, d | 4 | | |
| 3 | e, f | 4 | | |
| **Total** | | **10** | | |